In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


In [2]:
METHOD_NAME = "scaffold"
SCHEDULE_NAME = "continue"

RANDOM_SEED = 42
batch_size = 128
learning_rate = 1e-3
val_ratio = 0.1
prox_mu = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("device:", DEVICE)
print("torch version:", torch.__version__)
print("method:", METHOD_NAME, "| schedule:", SCHEDULE_NAME)


device: cuda
torch version: 2.10.0+cu130
method: scaffold | schedule: continue


In [3]:
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LeakyReLU(),
            nn.Linear(512, 1024),
            nn.LeakyReLU(),
            nn.Linear(1024, 512),
            nn.LeakyReLU(),
            nn.Linear(512, 256),
            nn.LeakyReLU(),
            nn.Linear(256, output_dim),
        )

    def forward(self, x):
        return self.net(x)


In [18]:
class ParaServer:
    def __init__(self, input_dim):
        self.input_dim = input_dim
        self.model = MLP(input_dim=input_dim).to(DEVICE)
        self.c_global = [torch.zeros_like(p, device=DEVICE) for p in self.model.parameters()]

    def upload(self, delta_ws, delta_cs):
        with torch.no_grad():
            for p, dw in zip(self.model.parameters(), delta_ws):
                p.add_(dw.to(DEVICE))
            for c_g, dc in zip(self.c_global, delta_cs):
                c_g.add_(dc.to(DEVICE))
        return self.download()

    def download(self):
        model = MLP(self.input_dim).to(DEVICE)
        model.load_state_dict(self.model.state_dict())
        return model, [c.detach().clone() for c in self.c_global]


In [19]:
def validate(model, X, y):
    model.eval()
    with torch.no_grad():
        y_pred = model(X)
        mse = torch.mean((y_pred - y) ** 2)
        rmse = torch.sqrt(mse)
        mae = torch.mean(torch.abs(y_pred - y))

        y_mean = torch.mean(y)
        denom = torch.sum((y - y_mean) ** 2)
        if torch.abs(denom) < 1e-12:
            r2 = torch.tensor(float("nan"), device=y.device)
        else:
            r2 = 1 - torch.sum((y_pred - y) ** 2) / denom

    return {
        "mse": float(mse.item()),
        "rmse": float(rmse.item()),
        "mae": float(mae.item()),
        "r2": float(r2.item()),
    }

In [20]:
class Node:
    def __init__(self, dsName, freq, input_dim, val_ratio=0.2, batch_size=batch_size):
        self.freq = freq
        self.input_dim = input_dim
        self.c_i = None

        dataset = pd.read_csv(dsName, encoding="utf-8").sample(
            frac=1, random_state=RANDOM_SEED
        ).reset_index(drop=True)

        X_all = dataset.loc[:, "freq":"L4"].to_numpy(dtype=np.float32)
        y_all = dataset.loc[:, "S11r":"S41i"].to_numpy(dtype=np.float32)

        n_total = X_all.shape[0]
        n_val = max(1, int(n_total * val_ratio))
        n_train = n_total - n_val

        self.X_train = X_all[:n_train]
        self.y_train = y_all[:n_train]
        self.X_vali = X_all[n_train:]
        self.y_vali = y_all[n_train:]

        self.X_train_t = torch.from_numpy(self.X_train).to(DEVICE)
        self.y_train_t = torch.from_numpy(self.y_train).to(DEVICE)

        self.X_vali_t = torch.from_numpy(self.X_vali).to(DEVICE)
        self.y_vali_t = torch.from_numpy(self.y_vali).to(DEVICE)

        train_ds = TensorDataset(
            torch.from_numpy(self.X_train),
            torch.from_numpy(self.y_train),
        )
        self.train_loader = DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            drop_last=False,
        )

        print(
            "client {} | train:{} | val:{}".format(
                self.freq, self.X_train.shape[0], self.X_vali.shape[0]
            )
        )

    def train(self, ps, global_round=None):
        model, c_global = ps.download()

        if self.c_i is None:
            self.c_i = [torch.zeros_like(p, device=DEVICE) for p in model.parameters()]
        else:
            self.c_i = [ci.detach().clone() for ci in self.c_i]

        model.train()
        w_before = [p.detach().clone() for p in model.parameters()]
        local_steps = 0

        current_lr = learning_rate

        for X_batch, y_batch in self.train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            model.zero_grad(set_to_none=True)
            y_pred = model(X_batch)
            tr_mse = torch.mean((y_pred - y_batch) ** 2)
            tr_mse.backward()

            corrected_grads = []
            for p, ci, cg in zip(model.parameters(), self.c_i, c_global):
                grad = p.grad.detach().clone()
                ci = ci.detach().clone()
                cg = cg.detach().clone()
                corrected_grads.append(grad - ci + cg)

            with torch.no_grad():
                for p, g_corr in zip(model.parameters(), corrected_grads):
                    p.sub_(current_lr * g_corr)

            local_steps += 1

        w_after = [p.detach().clone() for p in model.parameters()]
        delta_w = [(wa - wb).detach().clone() for wa, wb in zip(w_after, w_before)]

        K = max(local_steps, 1)
        scale = 1.0 / (K * current_lr)

        new_c_i = []
        delta_c_i = []
        for ci_old, c_g, w_b, w_a in zip(self.c_i, c_global, w_before, w_after):
            ci_old = ci_old.detach().clone()
            c_g = c_g.detach().clone()
            w_b = w_b.detach().clone()
            w_a = w_a.detach().clone()

            ci_new = ci_old - c_g + scale * (w_b - w_a)
            ci_new = ci_new.detach().clone()

            new_c_i.append(ci_new)
            delta_c_i.append((ci_new - ci_old).detach().clone())

        self.c_i = [ci.detach().clone() for ci in new_c_i]
        ps.upload(delta_w, delta_c_i)

        train_metrics = validate(model, self.X_train_t, self.y_train_t)

        print("node:{} round:{}".format(self.freq, global_round))
        print(
            "train mse:{:.6f} rmse:{:.6f} mae:{:.6f} r2:{:.6f}".format(
                train_metrics["mse"],
                train_metrics["rmse"],
                train_metrics["mae"],
                train_metrics["r2"],
            )
        )
        return train_metrics["r2"], self.freq


In [ ]:
r2_train = []
r2_vali = []
round_owner = []

r2_client_val = {
    2.4: [],
    2.5: [],
    2.6: [],
}

In [22]:
test_dataset = pd.read_csv("Test.csv", encoding="utf-8").sample(
    frac=1, random_state=RANDOM_SEED
).reset_index(drop=True)

X_vali = test_dataset.loc[:, "freq":"L4"].to_numpy(dtype=np.float32)
y_vali = test_dataset.loc[:, "S11r":"S41i"].to_numpy(dtype=np.float32)

input_dim = X_vali.shape[1]

X_vali_t = torch.from_numpy(X_vali).to(DEVICE)
y_vali_t = torch.from_numpy(y_vali).to(DEVICE)

print("global test shape:", X_vali.shape, y_vali.shape)
print("input_dim:", input_dim)


global test shape: (4500, 13) (4500, 8)
input_dim: 13


In [23]:
ps = ParaServer(input_dim=input_dim)

nodeList = [
    Node("./24Train.csv", 2.4, input_dim=input_dim, val_ratio=val_ratio, batch_size=batch_size),
    Node("./25Train.csv", 2.5, input_dim=input_dim, val_ratio=val_ratio, batch_size=batch_size),
    Node("./26Train.csv", 2.6, input_dim=input_dim, val_ratio=val_ratio, batch_size=batch_size),
]


client 2.4 | train:25650 | val:2850
client 2.5 | train:25650 | val:2850
client 2.6 | train:25650 | val:2850


In [ ]:
TOTAL_ROUNDS = 600
orders = [0, 1, 2]

turn = [
    np.array([[0, 100], [300, 400]]),
    np.array([[100, 200], [400, 500]]),
    np.array([[200, 300], [500, 600]]),
]
SHUFFLE_ORDERS = False

for global_round in range(TOTAL_ROUNDS):
    if SHUFFLE_ORDERS:
        random.shuffle(orders)

    for client_idx in orders:
        for start_round, end_round in turn[client_idx]:
            if start_round <= global_round < end_round:
                cur_r2, owner = nodeList[client_idx].train(ps, global_round=global_round)
                r2_train.append(cur_r2)
                round_owner.append(owner)

    model, _ = ps.download()
    all_r2 = validate(model, X_vali_t, y_vali_t)["r2"]
    r2_vali.append(all_r2)

    for node in nodeList:
        node_r2 = validate(model, node.X_vali_t, node.y_vali_t)["r2"]
        r2_client_val[node.freq].append(node_r2)

    print(
        "per-client val r2 | 2.4GHz: {:.6f} | 2.5GHz: {:.6f} | 2.6GHz: {:.6f}".format(
            r2_client_val[2.4][-1],
            r2_client_val[2.5][-1],
            r2_client_val[2.6][-1],
        )
    )

node:2.4 round:0
train mse:0.092413 rmse:0.303995 mae:0.249418 r2:0.235695
per-client val r2 | 2.4GHz: 0.238945 | 2.5GHz: 0.208186 | 2.6GHz: 0.184141
node:2.4 round:1
train mse:0.088261 rmse:0.297088 mae:0.243759 r2:0.270031
per-client val r2 | 2.4GHz: 0.271869 | 2.5GHz: 0.241349 | 2.6GHz: 0.213280
node:2.4 round:2
train mse:0.086431 rmse:0.293991 mae:0.240041 r2:0.285170
per-client val r2 | 2.4GHz: 0.287123 | 2.5GHz: 0.255285 | 2.6GHz: 0.226151
node:2.4 round:3
train mse:0.084276 rmse:0.290303 mae:0.237518 r2:0.302992
per-client val r2 | 2.4GHz: 0.304497 | 2.5GHz: 0.273043 | 2.6GHz: 0.241374
node:2.4 round:4
train mse:0.083265 rmse:0.288556 mae:0.235703 r2:0.311356
per-client val r2 | 2.4GHz: 0.312915 | 2.5GHz: 0.279572 | 2.6GHz: 0.245487
node:2.4 round:5
train mse:0.082257 rmse:0.286806 mae:0.233853 r2:0.319685
per-client val r2 | 2.4GHz: 0.320694 | 2.5GHz: 0.287697 | 2.6GHz: 0.253240
node:2.4 round:6
train mse:0.081731 rmse:0.285886 mae:0.233438 r2:0.324039
per-client val r2 | 2.4GH

In [25]:
def calc_forgetting(r2_dict):
    forgetting = {}
    for freq, history in r2_dict.items():
        if len(history) == 0:
            forgetting[freq] = np.nan
        else:
            forgetting[freq] = float(np.max(history) - history[-1])
    return forgetting


forgetting = calc_forgetting(r2_client_val)

print("\n===== Forgetting Summary (per-client local val) =====")
for freq in [2.4, 2.5, 2.6]:
    print(
        "client {} GHz | best {:.6f} | final {:.6f} | forgetting {:.6f}".format(
            freq,
            np.max(r2_client_val[freq]),
            r2_client_val[freq][-1],
            forgetting[freq],
        )
    )

avg_forgetting = np.mean([forgetting[2.4], forgetting[2.5], forgetting[2.6]])
print("Average Forgetting: {:.6f}".format(avg_forgetting))


save_dir = Path("./forgetting_logs")
save_dir.mkdir(parents=True, exist_ok=True)

r24 = np.array(r2_client_val[2.4], dtype=np.float32)
r25 = np.array(r2_client_val[2.5], dtype=np.float32)
r26 = np.array(r2_client_val[2.6], dtype=np.float32)

owners = np.array(round_owner, dtype=np.float32)
global_r2 = np.array(r2_vali, dtype=np.float32) if len(r2_vali) > 0 else np.array([], dtype=np.float32)
train_r2 = np.array(r2_train, dtype=np.float32) if len(r2_train) > 0 else np.array([], dtype=np.float32)

n_rounds = min(len(owners), len(r24), len(r25), len(r26))
round_idx = np.arange(n_rounds, dtype=np.int32)

owners = owners[:n_rounds]
r24 = r24[:n_rounds]
r25 = r25[:n_rounds]
r26 = r26[:n_rounds]

save_path = save_dir / f"{METHOD_NAME}_{SCHEDULE_NAME}_forgetting_log.npz"

np.savez(
    save_path,
    method_name=np.array([METHOD_NAME]),
    schedule_name=np.array([SCHEDULE_NAME]),
    round_idx=round_idx,
    round_owner=owners,
    client_24_val_r2=r24,
    client_25_val_r2=r25,
    client_26_val_r2=r26,
    global_test_r2=global_r2,
    train_r2=train_r2,
    best_24=np.array([float(np.max(r24))], dtype=np.float32),
    final_24=np.array([float(r24[-1])], dtype=np.float32),
    forget_24=np.array([forgetting[2.4]], dtype=np.float32),
    best_25=np.array([float(np.max(r25))], dtype=np.float32),
    final_25=np.array([float(r25[-1])], dtype=np.float32),
    forget_25=np.array([forgetting[2.5]], dtype=np.float32),
    best_26=np.array([float(np.max(r26))], dtype=np.float32),
    final_26=np.array([float(r26[-1])], dtype=np.float32),
    forget_26=np.array([forgetting[2.6]], dtype=np.float32),
    avg_forgetting=np.array([avg_forgetting], dtype=np.float32),
)

print(f"Saved to: {save_path}")



===== Forgetting Summary (per-client local val) =====
client 2.4 GHz | best 0.562346 | final 0.455615 | forgetting 0.106731
client 2.5 GHz | best 0.572709 | final 0.545947 | forgetting 0.026763
client 2.6 GHz | best 0.584181 | final 0.566598 | forgetting 0.017583
Average Forgetting: 0.050359
Saved to: forgetting_logs\scaffold_continue_forgetting_log.npz
